# 03 — Preprocessing & Feature Engineering

  Prepare the dataset for clustering: filter, clean text, and generate three embedding types (TF-IDF, MiniLM, KaLM).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

raw_path = project_root / "data" / "raw" / "arxiv_papers.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_path)
print(f"Loaded: {len(df):,} papers")

Loaded: 226,879 papers


## Filter and Clean

In [2]:
from src.preprocess import (
    clean_abstracts,
    filter_papers,
    flag_surveys,
    lemmatize_abstracts,
)

df = filter_papers(df)
df = clean_abstracts(df)
df = flag_surveys(df)

## Clean Text

Two versions:
- `abstract_clean`: whitespace normalized (for neural models)
- `abstract_lemma`: lowercased, stopwords removed, lemmatized (for TF-IDF)

In [3]:
print(f"Papers: {len(df):,}")
print(f"Surveys flagged: {df['is_survey'].sum():,}")
print(f"\nSample embed_text:\n{df['embed_text'].iloc[0][:200]}...")

Papers: 181,294
Surveys flagged: 3,089

Sample embed_text:
MediX-R1: Open Ended Medical Reinforcement Learning. We introduce MediX-R1, an open-ended Reinforcement Learning (RL) framework for medical multimodal large language models (MLLMs) that enables clinic...


## Lemmatize for TF-IDF

In [4]:
df = lemmatize_abstracts(df)

print(f"Sample abstract_lemma:\n{df['abstract_lemma'].iloc[0][:200]}...")

Sample abstract_lemma:
introduce medix open ended reinforcement learning framework medical multimodal large language model mllm enable clinically ground free form answer multiple choice format medix fine tune baseline visio...


In [5]:
df.to_csv(processed_dir / "arxiv_clean.csv", index=False)
print(f"Saved {len(df):,} papers to {processed_dir / 'arxiv_clean.csv'}")

Saved 181,294 papers to /home/dino/dev/learning/iu-bsc/unsupervised-learning/arxiv-ai-trends/data/processed/arxiv_clean.csv


## TF-IDF Vectorization

In [ ]:
from scipy import sparse

from src.features import compute_tfidf

tfidf_matrix, tfidf_model = compute_tfidf(df["abstract_lemma"])

sparse.save_npz(processed_dir / "tfidf_vectors.npz", tfidf_matrix)
print(f"Vocabulary size: {len(tfidf_model.vocabulary_):,}")

## MiniLM Embeddings (BERT-family baseline)

In [7]:
from src.features import compute_embeddings

minilm_embeddings = compute_embeddings(
    df["embed_text"].tolist(),
    model_name="all-MiniLM-L6-v2",
    batch_size=256,
)

np.save(processed_dir / "minilm_embeddings.npy", minilm_embeddings)

/home/dino/miniforge3/envs/arxiv-trends/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████| 103/103 [00:00<00:00, 1077.92it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|█████████████████████████████████████████████████████████████████| 709/709 [04:44<00:00,  2.49it/s]


 ## Free GPU memory before loading KaLM

In [8]:
import gc

import torch

del minilm_embeddings
gc.collect()
torch.cuda.empty_cache()

## KaLM Embeddings

In [6]:
kalm_embeddings = compute_embeddings(
    df["embed_text"].tolist(),
    model_name="HIT-TMG/KaLM-embedding-multilingual-mini-instruct-v1.5",
    batch_size=64,
)

np.save(processed_dir / "kalm_embeddings.npy", kalm_embeddings)

/home/dino/miniforge3/envs/arxiv-trends/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|█████████████████████████████████████████████████████████████| 2833/2833 [2:18:03<00:00,  2.92s/it]


## Summary

  Preprocessed **181,294** papers (filtered from 226,879 to 8 target categories).

  **New columns:**
  - `embed_text`: title + abstract (used for neural embeddings)
  - `is_survey`: flag for survey/review papers (3,089 flagged)

  **Text cleaning:**
  - `abstract_clean`: whitespace normalized
  - `abstract_lemma`: lowercased, stopwords removed, lemmatized via spaCy

  **Embeddings generated:**
  - TF-IDF: 181,294 x 27,401 (sparse, from `abstract_lemma`)
  - MiniLM (all-MiniLM-L6-v2): 181,294 x 384 (from `embed_text` — title + abstract)
  - KaLM (KaLM-multilingual-mini-instruct-v1.5): 181,294 x 896 (from `embed_text` — title + abstract)

  All saved to `data/processed/`.